In [11]:
import tkinter as tk
from tkinter import messagebox
from tkinter import ttk
from collections import deque
import re


# ============================================================
# XỬ LÝ MA TRẬN
# ============================================================

def parse_matrix(text):
    lines = text.strip().splitlines()

    matrix = []
    robot_pos = None
    robot_count = 0

    for r, line in enumerate(lines):
        row = list(map(int, re.findall(r"\d+", line)))

        if not row:
            continue

        matrix.append(row)

        for c, value in enumerate(row):
            if value not in [0, 1, 2, 3]:
                raise ValueError("Ma trận chỉ được dùng các số 0, 1, 2, 3")

            if value == 2 or value == 3:
                robot_pos = (r, c)
                robot_count += 1

    if not matrix:
        raise ValueError("Ma trận không được rỗng")

    if robot_count != 1:
        raise ValueError("Ma trận phải có đúng 1 robot, dùng số 2 hoặc 3")

    col_count = len(matrix[0])

    for row in matrix:
        if len(row) != col_count:
            raise ValueError("Các hàng trong ma trận phải có cùng số cột")

    robot_r, robot_c = robot_pos

    dirt_grid = []

    for r in range(len(matrix)):
        dirt_row = []

        for c in range(len(matrix[0])):
            value = matrix[r][c]

            # Ô robot chỉ là vị trí robot, không tính là ô bẩn hay ô sạch
            if r == robot_r and c == robot_c:
                dirt_row.append(0)

            elif value == 1:
                dirt_row.append(1)

            else:
                dirt_row.append(0)

        dirt_grid.append(tuple(dirt_row))

    return (robot_r, robot_c, tuple(dirt_grid))


def is_goal(state):
    robot_r, robot_c, dirt_grid = state

    for row in dirt_grid:
        if 1 in row:
            return False

    return True


# ============================================================
# ĐẶT TÊN NODE A, B, C, D...
# ============================================================

def number_to_label(number):
    letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    result = ""

    while True:
        result = letters[number % 26] + result
        number = number // 26 - 1

        if number < 0:
            break

    return result


def get_label(state, node_name, node_order):
    if state not in node_name:
        label = number_to_label(len(node_name))
        node_name[state] = label
        node_order.append(state)

    return node_name[state]


# ============================================================
# CHUYỂN STATE THÀNH MA TRẬN TEXT
# ============================================================

def state_to_matrix_text(state):
    robot_r, robot_c, dirt_grid = state

    result = []

    for r in range(len(dirt_grid)):
        row_text = []

        for c in range(len(dirt_grid[0])):
            if r == robot_r and c == robot_c:
                row_text.append("x")
            else:
                row_text.append(str(dirt_grid[r][c]))

        result.append(" ".join(row_text))

    return "\n".join(result)


def state_to_node_text(state, node_name):
    label = node_name[state]
    matrix_text = state_to_matrix_text(state)

    return f"{label}\n{matrix_text}"


# ============================================================
# SINH NODE CON
# ============================================================

def get_neighbors(state, version):
    robot_r, robot_c, dirt_grid = state

    rows = len(dirt_grid)
    cols = len(dirt_grid[0])

    neighbors = []

    # Version 1: ưu tiên giống vở: Right, Down, Left, Up
    if version == 1:
        actions = ["Right", "Down", "Left", "Up"]

    # Version 2: thứ tự khác để so sánh
    else:
        actions = ["Up", "Down", "Left", "Right"]

    for action in actions:
        new_r = robot_r
        new_c = robot_c

        if action == "Right":
            if robot_c < cols - 1:
                new_c = robot_c + 1
            else:
                continue

        elif action == "Down":
            if robot_r < rows - 1:
                new_r = robot_r + 1
            else:
                continue

        elif action == "Left":
            if robot_c > 0:
                new_c = robot_c - 1
            else:
                continue

        elif action == "Up":
            if robot_r > 0:
                new_r = robot_r - 1
            else:
                continue

        # Tạo ma trận bụi mới
        new_grid = [list(row) for row in dirt_grid]

        # Khi robot đi tới ô bẩn, ô đó tự động thành sạch
        new_grid[new_r][new_c] = 0

        new_grid = tuple(tuple(row) for row in new_grid)

        new_state = (new_r, new_c, new_grid)

        neighbors.append((action, new_state))

    return neighbors


# ============================================================
# FORMAT FRONTIER, REACHED
# ============================================================

def format_reached(visited_order, node_name):
    if not visited_order:
        return "[]"

    labels = []

    for state in visited_order:
        labels.append(node_name[state])

    return "[" + ", ".join(labels) + "]"


def format_frontier_after(children_added, node_name):
    """
    Hiển thị trong cột Frontier:
    - ma trận đầy đủ
    - sau đó mới ghi action -> node
    """

    if not children_added:
        return "Không sinh thêm node mới"

    result = []

    for action, state in children_added:
        label = node_name[state]
        matrix_text = state_to_matrix_text(state)

        text = matrix_text + f"\n{action} -> {label}"
        result.append(text)

    return "\n\n".join(result)


def format_frontier_list(frontier, node_name, algo):
    if not frontier:
        return "[]"

    labels = []

    for item in frontier:
        state = item[0]
        labels.append(node_name[state])

    if algo == "BFS":
        return "Queue: [" + ", ".join(labels) + "]"
    else:
        return "Stack: [" + ", ".join(labels) + "]"


# ============================================================
# BFS CÓ TRACE
# ============================================================

def bfs_with_trace(start_state, version):
    queue = deque()
    queue.append((start_state, []))

    discovered = set()
    discovered.add(start_state)

    node_name = {}
    node_order = []

    get_label(start_state, node_name, node_order)

    visited_order = []

    trace = []
    step = 0

    while queue:
        current_state, path = queue.popleft()
        step += 1

        # Reached là các node đã được duyệt
        if current_state not in visited_order:
            visited_order.append(current_state)

        if is_goal(current_state):
            trace.append({
                "step": step,
                "node": state_to_node_text(current_state, node_name),
                "frontier": "GOAL",
                "reached": format_reached(visited_order, node_name)
            })

            return path, trace, node_name

        children_added = []

        for action, next_state in get_neighbors(current_state, version):
            if next_state not in discovered:
                discovered.add(next_state)

                get_label(next_state, node_name, node_order)

                queue.append((next_state, path + [(action, next_state)]))

                children_added.append((action, next_state))

        frontier_text = format_frontier_after(children_added, node_name)
        frontier_text += "\n\nFrontier hiện tại:\n"
        frontier_text += format_frontier_list(queue, node_name, "BFS")

        trace.append({
            "step": step,
            "node": state_to_node_text(current_state, node_name),
            "frontier": frontier_text,
            "reached": format_reached(visited_order, node_name)
        })

    return None, trace, node_name


# ============================================================
# DFS CÓ TRACE
# ============================================================

def dfs_with_trace(start_state, version, max_depth=25):
    stack = []
    stack.append((start_state, []))

    discovered = set()
    discovered.add(start_state)

    node_name = {}
    node_order = []

    get_label(start_state, node_name, node_order)

    visited_order = []

    trace = []
    step = 0

    while stack:
        current_state, path = stack.pop()
        step += 1

        # Reached là các node đã được duyệt
        if current_state not in visited_order:
            visited_order.append(current_state)

        if is_goal(current_state):
            trace.append({
                "step": step,
                "node": state_to_node_text(current_state, node_name),
                "frontier": "GOAL",
                "reached": format_reached(visited_order, node_name)
            })

            return path, trace, node_name

        if len(path) >= max_depth:
            trace.append({
                "step": step,
                "node": state_to_node_text(current_state, node_name),
                "frontier": "Cắt nhánh vì đạt max depth",
                "reached": format_reached(visited_order, node_name)
            })

            continue

        neighbors = get_neighbors(current_state, version)

        children_added = []

        for action, next_state in neighbors:
            if next_state not in discovered:
                discovered.add(next_state)

                get_label(next_state, node_name, node_order)

                children_added.append((action, next_state))

        # DFS dùng stack.
        # Muốn Step 2 duyệt Node B thì Node B phải nằm trên đỉnh stack.
        for action, next_state in reversed(children_added):
            stack.append((next_state, path + [(action, next_state)]))

        frontier_text = format_frontier_after(children_added, node_name)
        frontier_text += "\n\nFrontier hiện tại:\n"
        frontier_text += format_frontier_list(stack, node_name, "DFS")

        trace.append({
            "step": step,
            "node": state_to_node_text(current_state, node_name),
            "frontier": frontier_text,
            "reached": format_reached(visited_order, node_name)
        })

    return None, trace, node_name


# ============================================================
# GIAO DIỆN TKINTER
# ============================================================

class VacuumApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Máy hút bụi BFS / DFS")
        self.root.geometry("1250x780")

        self.start_state = None
        self.solution = []
        self.trace = []
        self.current_step = 0
        self.node_name = {}

        title = tk.Label(
            root,
            text="MÔ PHỎNG MÁY HÚT BỤI - BFS / DFS",
            font=("Arial", 17, "bold")
        )
        title.pack(pady=8)

        # ====================================================
        # KHUNG TRÊN
        # ====================================================

        top_frame = tk.Frame(root)
        top_frame.pack(pady=5)

        input_frame = tk.LabelFrame(
            top_frame,
            text="Nhập ma trận",
            padx=10,
            pady=10
        )
        input_frame.grid(row=0, column=0, padx=10)

        self.matrix_input = tk.Text(
            input_frame,
            width=28,
            height=6,
            font=("Consolas", 13)
        )
        self.matrix_input.pack()

        default_matrix = """3 0 1
0 1 0
1 0 0"""
        self.matrix_input.insert("1.0", default_matrix)

        note = tk.Label(
            top_frame,
            text=(
                "Quy ước nhập:\n"
                "0 = ô sạch\n"
                "1 = ô bẩn\n"
                "2 = robot ở ô sạch\n"
                "3 = robot ở ô bẩn\n\n"
                "Khi hiển thị:\n"
                "x = vị trí robot\n"
                "Ô x không ghi sạch/bẩn\n"
                "Robot đi tới ô bẩn thì ô đó tự sạch\n"
                "Node đầu tiên = A"
            ),
            justify="left",
            font=("Arial", 11)
        )
        note.grid(row=0, column=1, padx=10)

        option_frame = tk.LabelFrame(
            top_frame,
            text="Chọn thuật toán",
            padx=10,
            pady=10
        )
        option_frame.grid(row=0, column=2, padx=10)

        self.algo_var = tk.StringVar()
        self.algo_var.set("BFS 1")

        algo_options = ["BFS 1", "BFS 2", "DFS 1", "DFS 2"]

        self.algo_menu = tk.OptionMenu(
            option_frame,
            self.algo_var,
            *algo_options
        )
        self.algo_menu.config(width=18)
        self.algo_menu.pack(pady=5)

        tk.Label(
            option_frame,
            text=(
                "BFS/DFS 1: Right, Down, Left, Up\n"
                "BFS/DFS 2: Up, Down, Left, Right\n"
                "Không dùng bước Suck riêng"
            ),
            justify="left"
        ).pack()

        # ====================================================
        # BUTTON
        # ====================================================

        button_frame = tk.Frame(root)
        button_frame.pack(pady=8)

        tk.Button(
            button_frame,
            text="Chạy thuật toán",
            width=16,
            bg="#4CAF50",
            fg="white",
            command=self.solve
        ).grid(row=0, column=0, padx=5)

        tk.Button(
            button_frame,
            text="Bước trước",
            width=16,
            command=self.prev_step
        ).grid(row=0, column=1, padx=5)

        tk.Button(
            button_frame,
            text="Bước tiếp",
            width=16,
            command=self.next_step
        ).grid(row=0, column=2, padx=5)

        tk.Button(
            button_frame,
            text="Reset",
            width=16,
            bg="#f44336",
            fg="white",
            command=self.reset
        ).grid(row=0, column=3, padx=5)

        # ====================================================
        # TRẠNG THÁI VÀ ĐƯỜNG ĐI
        # ====================================================

        self.status_label = tk.Label(
            root,
            text="Nhấn 'Chạy thuật toán' để bắt đầu.",
            font=("Arial", 12),
            fg="blue"
        )
        self.status_label.pack(pady=5)

        self.path_label = tk.Label(
            root,
            text="Đường đi lời giải: ",
            font=("Arial", 11),
            wraplength=1150,
            justify="left"
        )
        self.path_label.pack(pady=5)

        # ====================================================
        # KHUNG GIỮA: MA TRẬN + BẢNG
        # ====================================================

        middle_frame = tk.Frame(root)
        middle_frame.pack(fill="both", expand=True, padx=10, pady=5)

        self.grid_frame = tk.LabelFrame(
            middle_frame,
            text="Ma trận mô phỏng",
            padx=10,
            pady=10
        )
        self.grid_frame.pack(side="left", fill="both", padx=10)

        table_frame = tk.LabelFrame(
            middle_frame,
            text="Bảng duyệt chi tiết: Node - Frontier - Reached",
            padx=5,
            pady=5
        )
        table_frame.pack(side="right", fill="both", expand=True, padx=10)

        style = ttk.Style()
        style.configure("Treeview", rowheight=150)

        columns = ("step", "node", "frontier", "reached")

        self.tree = ttk.Treeview(
            table_frame,
            columns=columns,
            show="headings",
            height=13
        )

        self.tree.heading("step", text="Step")
        self.tree.heading("node", text="Node")
        self.tree.heading("frontier", text="Frontier")
        self.tree.heading("reached", text="Reached")

        self.tree.column("step", width=50, anchor="center")
        self.tree.column("node", width=150)
        self.tree.column("frontier", width=430)
        self.tree.column("reached", width=180)

        y_scroll = ttk.Scrollbar(
            table_frame,
            orient="vertical",
            command=self.tree.yview
        )

        x_scroll = ttk.Scrollbar(
            table_frame,
            orient="horizontal",
            command=self.tree.xview
        )

        self.tree.configure(
            yscrollcommand=y_scroll.set,
            xscrollcommand=x_scroll.set
        )

        self.tree.grid(row=0, column=0, sticky="nsew")
        y_scroll.grid(row=0, column=1, sticky="ns")
        x_scroll.grid(row=1, column=0, sticky="ew")

        table_frame.rowconfigure(0, weight=1)
        table_frame.columnconfigure(0, weight=1)

    # ========================================================
    # VẼ MA TRẬN MÔ PHỎNG BÊN TRÁI
    # ========================================================

    def draw_grid(self, state):
        for widget in self.grid_frame.winfo_children():
            widget.destroy()

        robot_r, robot_c, dirt_grid = state

        for r in range(len(dirt_grid)):
            for c in range(len(dirt_grid[0])):
                dirt = dirt_grid[r][c]

                if r == robot_r and c == robot_c:
                    text = "x"
                    bg_color = "white"
                else:
                    if dirt == 1:
                        text = "1\nBẩn"
                        bg_color = "#ffab91"
                    else:
                        text = "0\nSạch"
                        bg_color = "#c8e6c9"

                cell = tk.Label(
                    self.grid_frame,
                    text=text,
                    width=10,
                    height=4,
                    bg=bg_color,
                    relief="solid",
                    borderwidth=1,
                    font=("Arial", 14, "bold")
                )

                cell.grid(row=r, column=c, padx=3, pady=3)

    # ========================================================
    # HIỂN THỊ TRACE VÀO BẢNG
    # ========================================================

    def show_trace_table(self):
        for item in self.tree.get_children():
            self.tree.delete(item)

        for row in self.trace:
            self.tree.insert(
                "",
                "end",
                values=(
                    row["step"],
                    row["node"],
                    row["frontier"],
                    row["reached"]
                )
            )

    # ========================================================
    # CHẠY THUẬT TOÁN
    # ========================================================

    def solve(self):
        try:
            text = self.matrix_input.get("1.0", tk.END)
            self.start_state = parse_matrix(text)

            algo = self.algo_var.get()

            if algo == "BFS 1":
                self.solution, self.trace, self.node_name = bfs_with_trace(
                    self.start_state,
                    version=1
                )

            elif algo == "BFS 2":
                self.solution, self.trace, self.node_name = bfs_with_trace(
                    self.start_state,
                    version=2
                )

            elif algo == "DFS 1":
                self.solution, self.trace, self.node_name = dfs_with_trace(
                    self.start_state,
                    version=1
                )

            elif algo == "DFS 2":
                self.solution, self.trace, self.node_name = dfs_with_trace(
                    self.start_state,
                    version=2
                )

            self.current_step = 0
            self.draw_grid(self.start_state)
            self.show_trace_table()

            if self.solution is None:
                self.status_label.config(
                    text=f"{algo}: Không tìm thấy lời giải."
                )
                self.path_label.config(
                    text="Đường đi lời giải: Không có"
                )
                return

            path_text = []
            current_state = self.start_state
            current_label = self.node_name[current_state]

            for action, next_state in self.solution:
                next_label = self.node_name[next_state]
                path_text.append(f"{current_label} --{action}--> {next_label}")

                current_state = next_state
                current_label = next_label

            self.status_label.config(
                text=f"{algo}: Tìm thấy lời giải gồm {len(self.solution)} bước. Node đầu là A."
            )

            self.path_label.config(
                text="Đường đi lời giải: " + " | ".join(path_text)
            )

        except Exception as e:
            messagebox.showerror("Lỗi", str(e))

    # ========================================================
    # BƯỚC TIẾP
    # ========================================================

    def next_step(self):
        if self.start_state is None or not self.solution:
            messagebox.showwarning(
                "Thông báo",
                "Bạn cần chạy thuật toán trước."
            )
            return

        if self.current_step < len(self.solution):
            action, state = self.solution[self.current_step]
            self.current_step += 1

            self.draw_grid(state)

            label = self.node_name[state]

            self.status_label.config(
                text=f"Bước {self.current_step}: {action} → Node {label}"
            )

        else:
            self.status_label.config(
                text="Đã hoàn thành. Tất cả ô đã sạch."
            )

    # ========================================================
    # BƯỚC TRƯỚC
    # ========================================================

    def prev_step(self):
        if self.start_state is None or not self.solution:
            messagebox.showwarning(
                "Thông báo",
                "Bạn cần chạy thuật toán trước."
            )
            return

        if self.current_step > 0:
            self.current_step -= 1

            if self.current_step == 0:
                self.draw_grid(self.start_state)
                self.status_label.config(text="Bước 0: Node A")
            else:
                action, state = self.solution[self.current_step - 1]
                self.draw_grid(state)

                label = self.node_name[state]

                self.status_label.config(
                    text=f"Bước {self.current_step}: {action} → Node {label}"
                )

    # ========================================================
    # RESET
    # ========================================================

    def reset(self):
        self.start_state = None
        self.solution = []
        self.trace = []
        self.current_step = 0
        self.node_name = {}

        for widget in self.grid_frame.winfo_children():
            widget.destroy()

        for item in self.tree.get_children():
            self.tree.delete(item)

        self.status_label.config(
            text="Đã reset. Nhấn 'Chạy thuật toán' để bắt đầu."
        )

        self.path_label.config(
            text="Đường đi lời giải: "
        )


# ============================================================
# MAIN
# ============================================================

root = tk.Tk()

root.lift()
root.attributes("-topmost", True)
root.after(1000, lambda: root.attributes("-topmost", False))

app = VacuumApp(root)

root.mainloop()